# Muraqam (مُرقّم) — single AraBERT + **double forward pass** (+ self-conditioning)

Stripped back to **one AraBERT** so you can read a clean single-model signal before scaling up, plus the two ideas from the chat:

1. **Double forward pass (inference).** Pass 1 predicts on the bare word stream. We then materialize **only the high-confidence reliable marks** (`.` `،`) back into the text and run **pass 2**, so the encoder now sees explicit sentence boundaries. Macro-F1 is pinned by the structural classes (`؛ ! ؟ :`) whose placement is a *boundary-rank* decision — pass 2 conditions exactly those on the recovered structure. This is a denoising / deliberation step, not diffusion.
2. **Self-conditioning (training).** Pass-2 input is technically OOD for a head fine-tuned only on stripped text. So during training we append a **noisy subset of gold reliable marks** to a fraction of samples (curriculum `0 → MAX_P`), so the head *learns to use* injected structure. Toggle `SELF_COND_TRAIN=False` to A/B a plain model that only does two-pass at inference.

**What to read after a run:** the validation cell prints single-pass vs two-pass **macro-F1 and per-class F1 side by side**. The gain should be concentrated on `؛ / ! / ؟ / :`. If it's smeared or negative, your `INJECT_THRESH` is too loose and you're propagating pass-1 dot errors — raise it.

**Key knobs (cell 2):** `WARM_START` (punctuation-pretrained AraBERT vs plain AraBERTv2), `TWO_PASS`, `INJECT_MARKS`, `INJECT_THRESH` (precision gate, keep high ~0.8), `PASS2_BLEND`, and the `SELF_COND_*` schedule. To go back to the ensemble later, just add CAMeLBERT back into `MODELS`.

Word count is preserved throughout (marks are **appended to a word**, never added as new word units), so the host-metric alignment and label indexing stay exact. The honorific `-` is never injected — it stays rule-derived.

> Requires **internet ON** and a **GPU**.


In [ ]:
# =========================================================================
#  Muraqam (مُرقّم) — Arabic Punctuation Restoration
#  Single AraBERT + double forward pass (+ optional self-conditioning).
#  Metric: macro-F1 over 7 marks ( . ، ؟ ! : ؛ - ), multi-label per gap.
# =========================================================================
!pip install -q transformers torch

import os, re, random, math, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- data paths (Kaggle) ---
TRAIN_CSV = "/kaggle/input/competitions/muraqqamchallenge/train.csv"   # adjust to your slug
TEST_CSV  = "/kaggle/input/competitions/muraqqamchallenge/test.csv"
if not os.path.exists(TRAIN_CSV):                         # fallback for local
    TRAIN_CSV = "train.csv"; TEST_CSV = "test.csv"

# --- the 7 scored marks, fixed canonical order ---
MARKS = ['.', '،', '؟', '!', ':', '؛', '-']
M2I = {m: i for i, m in enumerate(MARKS)}
NUM_MARKS = len(MARKS)

# =========================================================================
#  SINGLE MODEL first — read one AraBERT's signal before going hard.
#  WARM_START=True  -> makdadTaleb/arabic-punctuation-arabert   (AraBERT already
#                      fine-tuned on ~400k UN sentences for punctuation; we load
#                      its ENCODER via AutoModel and attach our multi-label head).
#  WARM_START=False -> aubmindlab/bert-base-arabertv2           (plain AraBERT).
#  To return to the ensemble later, add CAMeLBERT back into MODELS.
# =========================================================================
WARM_START = True
ARABERT_MEMBER = ("makdadTaleb/arabic-punctuation-arabert" if WARM_START
                  else "aubmindlab/bert-base-arabertv2")
MODELS = [ARABERT_MEMBER]          # <- single AraBERT (was CAMeLBERT + AraBERT)

# --- training config ---
CHUNK_WORDS = 180     # words per window (docs are long -> sliding window)
STRIDE      = 140     # overlap = 40 words
MAX_LEN     = 320     # subword cap per window
BATCH       = 8
EPOCHS      = 20
LR          = 2e-5
FOCAL_GAMMA = 2.0     # focal loss: focus on hard/rare marks
VAL_FRAC    = 0.10
USE_AMP             = True
EARLY_STOP_PATIENCE = 5
MAX_EPOCHS_CAP      = 20

# --- honorifics wrapped as -X- (rule-handled; never injected) ---
HONORIFICS = {"ﷺ"}
FORCE_HONORIFIC_RULE = True

# =========================================================================
#  DOUBLE FORWARD PASS (inference-time refinement).
#  Pass 1 predicts on the bare word stream. We then MATERIALIZE only the
#  high-confidence reliable marks (periods, commas) back into the text and run
#  Pass 2, so the encoder now sees explicit sentence boundaries. This targets
#  the macro-F1 bottleneck classes (؛ ! ؟ :), whose placement is a structural
#  decision that needs to know where sentences end.
# =========================================================================
TWO_PASS      = True
INJECT_MARKS  = ['.', '،']            # only these get injected into pass-2 input
INJECT_THRESH = 0.80                  # HIGH precision gate (NOT the tuned decode thr)
PASS2_BLEND   = "reliable_from_p1"    # {"p2", "mean", "reliable_from_p1"}

# =========================================================================
#  SELF-CONDITIONING (training-time) -> makes pass-2 input IN-distribution.
#  With prob inject_p (curriculum 0 -> MAX_P over WARMUP epochs) we append a
#  NOISY subset of the GOLD reliable marks to the input words during training,
#  so the head learns to exploit injected structure instead of treating it as
#  OOD. Noise (drop kept marks / add spurious ones) mimics imperfect pass-1
#  output and blocks the trivial copy shortcut. False -> two-pass at inference
#  only, on a plain model (good A/B baseline).
# =========================================================================
SELF_COND_TRAIN         = True
SELF_COND_MAX_P         = 0.50   # max fraction of training samples that get injection
SELF_COND_WARMUP_EPOCHS = 3      # ramp inject_p to MAX_P over this many epochs
SELF_COND_KEEP          = 0.90   # keep a gold reliable mark (simulate pass-1 recall)
SELF_COND_ADD           = 0.03   # add a spurious reliable mark (simulate pass-1 FPs)

print("device:", device, "| model(s):", MODELS)
print(f"two_pass={TWO_PASS} | self_cond={SELF_COND_TRAIN} | inject={INJECT_MARKS} @>={INJECT_THRESH} | blend={PASS2_BLEND}")


## 1. Host metric (verbatim)

In [ ]:
# Host metric (verbatim) — for trustworthy LOCAL validation before you submit.
class ParticipantVisibleError(Exception):
    pass

"""
Kaggle metric for Arabic Punctuation Restoration.

Conventions:
    - `solution` is the full test CSV, including the hidden gold column.
    - `submission` is the competitor's CSV.
    - Both have a row_id column (already aligned & sorted by Kaggle).
    - `solution` has a column `raw` with the unpunctuated input and a column
      `gold` with the reference punctuated string.
    - `submission` has a column `prediction` with the competitor's
      punctuated string.

Scoring:
    Macro-F1 over the 7 Arabic sentence-punctuation classes
    ( . ، ؟ ! : ؛ - ), EXCLUDING the "no punctuation" class.

"""

import re
from typing import Optional

import pandas as pd
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------------------------------------------------------------------
# Classical Arabic punctuation-restoration set. Anything outside this is
# treated as word content (e.g. parentheses, quotes, numbers) and is NOT
# scored. Competitors must preserve those characters in their predictions.
# ---------------------------------------------------------------------------
VALID_SYMBOLS = set('.،؟!:؛-')


def _tokenize_gold(text: str):
    """
    Split a (gold or prediction) string into (leading_gap, [(word, trailing_gap), ...]).
    A "word" is a maximal run of non-whitespace, non-whitelist chars.
    A "gap" is any run of whitelist chars between words.
    """
    leading_gap_chars = []
    pairs = []
    current_word_chars = []
    in_word = False

    for ch in text:
        if ch.isspace():
            if in_word:
                pairs.append([''.join(current_word_chars), []])
                current_word_chars = []
                in_word = False
            continue

        if ch in VALID_SYMBOLS:
            if in_word:
                pairs.append([''.join(current_word_chars), [ch]])
                current_word_chars = []
                in_word = False
            else:
                if pairs:
                    pairs[-1][1].append(ch)
                else:
                    leading_gap_chars.append(ch)
            continue

        if not in_word:
            in_word = True
            current_word_chars = [ch]
        else:
            current_word_chars.append(ch)

    if in_word:
        pairs.append([''.join(current_word_chars), []])

    return ''.join(leading_gap_chars), [(w, ''.join(g)) for (w, g) in pairs]


def _extract_labels(raw_text: str, generated_text: str, role: str):
    """
    Align `generated_text` to `raw_text` word-by-word and return one list of
    symbols per word, representing the punctuation that appears in the gap
    after each word.

    Raises ValueError on any structural mismatch.
    """
    if raw_text is None or generated_text is None:
        raise ValueError(f"[{role}] text is empty or null")

    raw_words = str(raw_text).strip().split()
    if not raw_words:
        raise ValueError(f"[{role}] raw text contains no words")

    _, pairs = _tokenize_gold(str(generated_text))
    gen_words = [w for (w, _) in pairs]

    if len(gen_words) != len(raw_words):
        raise ValueError(
            f"[{role}] word-count mismatch: raw has {len(raw_words)} words, "
            f"{role} has {len(gen_words)}"
        )

    for i, (rw, gw) in enumerate(zip(raw_words, gen_words)):
        if rw != gw:
            raise ValueError(
                f"[{role}] word mismatch at position {i}: "
                f"raw='{rw}' vs {role}='{gw}'"
            )

    labels = []
    for _, gap in pairs:
        syms = [c for c in gap if c in VALID_SYMBOLS]
        labels.append(syms if syms else ['0'])

    return labels


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    raw_column_name: str = 'text',
    gold_column_name: str = 'final_text',
    prediction_column_name: str = 'final_text',
) -> float:
    """
    Returns macro-F1 over the 7 Arabic punctuation
    classes, excluding the "no punctuation" class.
    """
    # --- 0. Drop row_id; Kaggle has already aligned the two frames -----------
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # --- 1. Column presence -------------------------------------------------
    for col in (raw_column_name, gold_column_name):
        if col not in solution.columns:
            # Organizer-side problem — hidden from competitor.
            raise RuntimeError(f"Solution is missing column '{col}'")
    if prediction_column_name not in submission.columns:
        raise ParticipantVisibleError(
            f"Submission is missing column '{prediction_column_name}'"
        )

    # --- 2. Length match ----------------------------------------------------
    if len(solution) != len(submission):
        raise ParticipantVisibleError(
            f"Submission has {len(submission)} rows, expected {len(solution)}"
        )

    # --- 3. Extract labels row-by-row --------------------------------------
    true_labels = []
    pred_labels = []

    raws = solution[raw_column_name].tolist()
    golds = solution[gold_column_name].tolist()
    preds = submission[prediction_column_name].tolist()

    for idx, (raw, gold, pred) in enumerate(zip(raws, golds, preds)):
        try:
            gold_seq = _extract_labels(raw, gold, role="gold")
        except ValueError as e:
            # Organizer-side: our own gold CSV is malformed for this row.
            raise RuntimeError(
                f"Gold extraction failed on row index {idx}: {e}"
            ) from e

        try:
            pred_seq = _extract_labels(raw, pred, role="prediction")
        except ValueError as e:
            # Competitor can fix this themselves.
            raise ParticipantVisibleError(
                f"Prediction at row index {idx} does not align with the raw "
                f"input. Your prediction must contain the same sequence of "
                f"non-punctuation words as the input, with only the allowed "
                f"punctuation symbols {sorted(VALID_SYMBOLS)} inserted "
                f"between them. Details: {e}"
            ) from e

        if len(gold_seq) != len(pred_seq):
            raise ParticipantVisibleError(
                f"Row {idx}: prediction has {len(pred_seq)} word positions "
                f"but the input has {len(gold_seq)}"
            )

        true_labels.extend(gold_seq)
        pred_labels.extend(pred_seq)

    # --- 4. Binarize using gold ∪ pred so hallucinated marks cost precision --
    all_classes = sorted(VALID_SYMBOLS) + ['0']
    mlb = MultiLabelBinarizer(classes=all_classes)
    y_true = mlb.fit_transform(true_labels)
    y_pred = mlb.transform(pred_labels)

    classes = list(mlb.classes_)
    zero_idx = classes.index('0')
    scored_cols = [i for i in range(len(classes)) if i != zero_idx]

    return float(f1_score(
        y_true[:, scored_cols],
        y_pred[:, scored_cols],
        average='macro',
        zero_division=0,
    ))

## 2. Word-gap tokenizer, labeling & honorific rule

In [ ]:
# =========================================================================
#  Word-gap tokenizer (IDENTICAL to the host metric) + labeling + rules
#  Sharing the metric's tokenizer guarantees zero train/scoring gap.
# =========================================================================
VALID_SYMBOLS = set('.،؟!:؛-')

def tokenize_gold(text):
    """-> (leading_gap, [(word, trailing_gap), ...]).  Matches host metric."""
    leading=[]; pairs=[]; cur=[]; in_word=False
    for ch in str(text):
        if ch.isspace():
            if in_word: pairs.append([''.join(cur), []]); cur=[]; in_word=False
            continue
        if ch in VALID_SYMBOLS:
            if in_word: pairs.append([''.join(cur), [ch]]); cur=[]; in_word=False
            else:
                if pairs: pairs[-1][1].append(ch)
                else: leading.append(ch)
            continue
        if not in_word: in_word=True; cur=[ch]
        else: cur.append(ch)
    if in_word: pairs.append([''.join(cur), []])
    return ''.join(leading), [(w, ''.join(g)) for w,g in pairs]

def gold_to_labels(final_text):
    """Return (words, Y) where Y is (n_words, NUM_MARKS) multi-hot of the gap AFTER each word."""
    _, pairs = tokenize_gold(final_text)
    words=[w for w,_ in pairs]
    Y=np.zeros((len(words), NUM_MARKS), dtype=np.float32)
    for i,(_,gap) in enumerate(pairs):
        for c in gap:
            if c in M2I: Y[i, M2I[c]] = 1.0
    return words, Y

def words_of_raw(raw):
    """Word units exactly as the metric derives them: whitespace split."""
    return str(raw).strip().split()

MARK_ORDER = MARKS  # canonical write order inside a gap (scoring is set-based anyway)

def reconstruct(words, pred_multi):
    """words + per-word multi-hot -> final_text string (word count preserved)."""
    toks=[]
    for w, row in zip(words, pred_multi):
        marks=''.join(m for m in MARK_ORDER if row[M2I[m]]>0)
        toks.append(w+marks)
    return ' '.join(toks)

def apply_honorific_rule(words, pred_multi):
    """Force -X- around each honorific: dash on its own gap AND the previous word's gap."""
    if not FORCE_HONORIFIC_RULE: return pred_multi
    P=pred_multi.copy()
    di=M2I['-']
    for i,w in enumerate(words):
        if w in HONORIFICS:
            P[i, di]=1.0
            if i>0: P[i-1, di]=1.0
    return P

# ---- load + split ----
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=["text","final_text"]).reset_index(drop=True)
idx = np.arange(len(df)); rng=np.random.default_rng(SEED); rng.shuffle(idx)
n_val=max(1,int(len(df)*VAL_FRAC))
val_idx=set(idx[:n_val].tolist())
train_rows=[df.iloc[i] for i in range(len(df)) if i not in val_idx]
val_rows  =[df.iloc[i] for i in range(len(df)) if i in val_idx]
print(f"train rows: {len(train_rows)} | val rows: {len(val_rows)}")

# sanity: gold words must equal raw words (they do, per analysis)
_bad=0
for r in train_rows+val_rows:
    w_gold,_=gold_to_labels(r["final_text"]); 
    if w_gold!=words_of_raw(r["text"]): _bad+=1
print(f"word-alignment mismatches (want 0): {_bad}")

## 3. Sliding-window dataset

In [ ]:
# =========================================================================
#  Sliding-window dataset. Punctuation is predicted at the LAST subword of
#  each word. With SELF_COND_TRAIN, a fraction of samples get a NOISY subset of
#  gold reliable marks appended to their input words, so the model learns to use
#  the structure that pass-2 will inject at inference. Word COUNT is preserved
#  (marks are appended to a word, never added as new word units), so host-metric
#  alignment and label indexing stay exact.
# =========================================================================
INJECT_IDX = [M2I[m] for m in INJECT_MARKS]

def chunk_words(words, Y=None):
    """Yield (word_slice, Y_slice, start_index) windows over a long word list."""
    n=len(words)
    if n<=CHUNK_WORDS:
        yield words, (Y if Y is not None else None), 0; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS, n)
        yield words[s:e], (Y[s:e] if Y is not None else None), s
        if e==n: break
        s+=STRIDE

class PunctDataset(Dataset):
    def __init__(self, rows, tokenizer, has_labels=True):
        self.samples=[]
        self.tok=tokenizer; self.has_labels=has_labels
        self.inject_p=0.0                              # set per-epoch by the trainer
        # headroom so injected punctuation subwords don't truncate trailing words
        self.max_len=MAX_LEN + (64 if (has_labels and SELF_COND_TRAIN) else 0)
        for r in rows:
            words=words_of_raw(r["text"])
            if has_labels:
                gw,Y=gold_to_labels(r["final_text"])   # words already == gw (verified)
            else:
                Y=None
            for wslice,Yslice,start in chunk_words(words,Y):
                self.samples.append((wslice,Yslice))
    def __len__(self): return len(self.samples)

    def _inject(self, words, Y):
        """Append a NOISY subset of gold reliable marks to words (pass-2 sim)."""
        out=[]
        for i,w in enumerate(words):
            s=w
            for mi in INJECT_IDX:
                has = Y[i,mi] > 0
                if has and random.random() < SELF_COND_KEEP:            s += MARKS[mi]
                elif (not has) and random.random() < SELF_COND_ADD:     s += MARKS[mi]
            out.append(s)
        return out

    def __getitem__(self,i):
        words,Y=self.samples[i]
        enc_words=words
        if self.has_labels and Y is not None and self.inject_p>0.0 and random.random()<self.inject_p:
            enc_words=self._inject(words,Y)
        enc=self.tok(enc_words, is_split_into_words=True, truncation=True,
                     max_length=self.max_len, return_tensors=None)
        word_ids=enc.word_ids()
        # mark the LAST subword of each word as the active prediction position
        last_pos={}
        for pos,wid in enumerate(word_ids):
            if wid is not None: last_pos[wid]=pos
        active=np.zeros(len(word_ids),dtype=bool)
        labels=np.zeros((len(word_ids),NUM_MARKS),dtype=np.float32)
        wid_at=np.full(len(word_ids),-1,dtype=np.int64)
        for wid,pos in last_pos.items():
            active[pos]=True; wid_at[pos]=wid
            if Y is not None and wid<len(Y): labels[pos]=Y[wid]
        return {"input_ids":enc["input_ids"],"attention_mask":enc["attention_mask"],
                "active":active,"labels":labels,"word_ids":wid_at}

def collate(batch, pad_id):
    maxlen=max(len(b["input_ids"]) for b in batch)
    B=len(batch)
    input_ids=np.full((B,maxlen),pad_id,dtype=np.int64)
    attn=np.zeros((B,maxlen),dtype=np.int64)
    active=np.zeros((B,maxlen),dtype=bool)
    labels=np.zeros((B,maxlen,NUM_MARKS),dtype=np.float32)
    wids=np.full((B,maxlen),-1,dtype=np.int64)
    for i,b in enumerate(batch):
        L=len(b["input_ids"])
        input_ids[i,:L]=b["input_ids"]; attn[i,:L]=b["attention_mask"]
        active[i,:L]=b["active"]; labels[i,:L]=b["labels"]; wids[i,:L]=b["word_ids"]
    return (torch.tensor(input_ids),torch.tensor(attn),torch.tensor(active),
            torch.tensor(labels),torch.tensor(wids))
print("dataset utilities ready | self-conditioning:", SELF_COND_TRAIN)


## 4. Multi-label model + focal loss
(`PunctModel` uses `AutoModel.from_pretrained`, which loads the warm-start encoder and discards its token-classification head.)

In [ ]:
# =========================================================================
#  Multi-LABEL token classifier (independent sigmoid per mark) + focal loss.
#  Multi-label (NOT 8-way softmax) is essential: gaps like ؟! carry two marks,
#  and the metric scores each mark independently. Softmax would forfeit them.
# =========================================================================
class PunctModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone=AutoModel.from_pretrained(model_name)
        h=self.backbone.config.hidden_size
        self.drop=nn.Dropout(0.1)
        self.head=nn.Linear(h,NUM_MARKS)
    def forward(self,input_ids,attention_mask):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        return self.head(self.drop(out))          # (B,T,NUM_MARKS) logits

def focal_bce(logits, targets, active, gamma=FOCAL_GAMMA, pos_weight=None):
    """Focal binary cross-entropy, averaged over ACTIVE positions only."""
    logits=logits[active]; targets=targets[active]          # (N,NUM_MARKS)
    if logits.numel()==0:
        return logits.sum()*0.0
    bce=nn.functional.binary_cross_entropy_with_logits(
        logits,targets,reduction='none',pos_weight=pos_weight)
    p=torch.sigmoid(logits)
    p_t=p*targets+(1-p)*(1-targets)
    focal=((1-p_t)**gamma)*bce
    return focal.mean()

# per-mark positive weight from class frequency (rarer -> upweighted)
def compute_pos_weight(rows):
    pos=np.zeros(NUM_MARKS); tot=0
    for r in rows:
        _,Y=gold_to_labels(r["final_text"]); pos+=Y.sum(0); tot+=len(Y)
    neg=tot-pos
    w=np.clip(neg/np.clip(pos,1,None),1.0,20.0)   # cap to avoid instability
    return torch.tensor(w,dtype=torch.float32)
print("model + focal loss ready")

## 5. Training (AMP + early stopping) & windowed inference

In [ ]:
# =========================================================================
#  Training with AMP + EARLY STOPPING on validation macro-F1, then SINGLE-pass
#  and TWO-pass windowed inference. Early-stopping signal stays single-pass
#  (measures base model quality); two-pass refinement is evaluated after.
# =========================================================================
from functools import partial

def _val_macro_f1(model, tok, val_rows):
    """Cheap per-word macro-F1 at 0.5 over the 7 marks (early-stopping signal)."""
    model.eval(); pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tp=np.zeros(NUM_MARKS); fp=np.zeros(NUM_MARKS); fn=np.zeros(NUM_MARKS)
    with torch.no_grad():
        for r in val_rows:
            words=words_of_raw(r["text"]); n=len(words)
            _,Y=gold_to_labels(r["final_text"])
            acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
            for wslice,_,start in chunk_words(words,None):
                enc=tok(wslice,is_split_into_words=True,truncation=True,max_length=MAX_LEN,return_tensors="pt")
                wid=enc.word_ids()
                logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
                probs=torch.sigmoid(logits).float().cpu().numpy()
                last={}
                for pos,w in enumerate(wid):
                    if w is not None: last[w]=pos
                for w,pos in last.items():
                    gi=start+w
                    if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
            pred=((acc/cnt)>=0.5).astype(int)
            tp+=((pred==1)&(Y==1)).sum(0); fp+=((pred==1)&(Y==0)).sum(0); fn+=((pred==0)&(Y==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0,2*prec*rec/np.clip(prec+rec,1e-9,None),0.0)
    return float(f1.mean())

def train_one(model_name, train_rows, val_rows):
    tok=AutoTokenizer.from_pretrained(model_name)
    pad_id=tok.pad_token_id if tok.pad_token_id is not None else 0
    tr=PunctDataset(train_rows,tok,has_labels=True)
    dl=DataLoader(tr,batch_size=BATCH,shuffle=True,collate_fn=partial(collate,pad_id=pad_id))
    model=PunctModel(model_name).to(device)     # AutoModel loads encoder; warm-start head discarded
    pw=compute_pos_weight(train_rows).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    n_epochs=MAX_EPOCHS_CAP
    total=len(dl)*n_epochs
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,total_steps=total,pct_start=0.1)
    scaler=torch.cuda.amp.GradScaler(enabled=(USE_AMP and device=="cuda"))

    best_f1=-1.0; best_state=None; patience=0
    for ep in range(n_epochs):
        # self-conditioning curriculum: ramp inject_p 0 -> MAX_P over WARMUP epochs
        if SELF_COND_TRAIN:
            tr.inject_p = SELF_COND_MAX_P * min(1.0, ep / max(1, SELF_COND_WARMUP_EPOCHS))
        model.train(); run=0.0
        for input_ids,attn,active,labels,_ in dl:
            input_ids,attn=input_ids.to(device),attn.to(device)
            active,labels=active.to(device),labels.to(device)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(USE_AMP and device=="cuda")):
                logits=model(input_ids,attn)
                loss=focal_bce(logits,labels,active,pos_weight=pw)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); sched.step(); run+=loss.item()
        vf1=_val_macro_f1(model,tok,val_rows) if val_rows else -1.0
        tag=f"  [{model_name.split('/')[-1]}] epoch {ep+1}/{n_epochs} loss {run/len(dl):.4f}"
        if SELF_COND_TRAIN: tag+=f" | inject_p {tr.inject_p:.2f}"
        if val_rows:
            tag+=f" | val macroF1 {vf1:.4f}"
            if vf1>best_f1+1e-4:
                best_f1=vf1; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience=0; tag+="  *"
            else:
                patience+=1
        print(tag)
        if val_rows and patience>=EARLY_STOP_PATIENCE:
            print(f"    early stop (no val gain in {EARLY_STOP_PATIENCE} epochs); best macroF1 {best_f1:.4f}")
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, tok

# ---------------------------------------------------------------------------
#  Inference: single pass, plus TWO-PASS refinement.
# ---------------------------------------------------------------------------
def _chunk_bounds(n):
    if n<=CHUNK_WORDS:
        yield 0,n; return
    s=0
    while s<n:
        e=min(s+CHUNK_WORDS,n); yield s,e
        if e==n: break
        s+=STRIDE

@torch.no_grad()
def _infer_rows(model, tok, rows, enc_words_list=None):
    """Windowed inference. If enc_words_list is given (one list per row, SAME
    length as that row's words), those strings are fed to the encoder while
    predictions are still read back onto the ORIGINAL word positions."""
    model.eval()
    results=[]
    for ri,r in enumerate(rows):
        words=words_of_raw(r["text"]); n=len(words)
        enc_words = enc_words_list[ri] if enc_words_list is not None else words
        ml = MAX_LEN + (64 if enc_words_list is not None else 0)   # headroom for injected subwords
        acc=np.zeros((n,NUM_MARKS)); cnt=np.zeros((n,1))+1e-9
        for s,e in _chunk_bounds(n):
            ew=enc_words[s:e]
            enc=tok(ew,is_split_into_words=True,truncation=True,max_length=ml,return_tensors="pt")
            wid=enc.word_ids()
            logits=model(enc["input_ids"].to(device),enc["attention_mask"].to(device))[0]
            probs=torch.sigmoid(logits).float().cpu().numpy()
            last={}
            for pos,w in enumerate(wid):
                if w is not None: last[w]=pos
            for w,pos in last.items():
                gi=s+w
                if gi<n: acc[gi]+=probs[pos]; cnt[gi]+=1
        results.append((words, acc/cnt))
    return results

def predict_probs(model, tok, rows):
    """Single forward pass (original behaviour)."""
    return _infer_rows(model, tok, rows, enc_words_list=None)

def _build_injection(rows, pass1_results):
    """From pass-1 probs, materialize ONLY the high-confidence reliable marks
    (INJECT_MARKS) into the word stream: word -> word+mark. Never '-' (honorific
    dash is rule-derived)."""
    inj=[]
    for r,(words,P) in zip(rows, pass1_results):
        ew=[]
        for i,w in enumerate(words):
            s=w
            for mi,m in zip(INJECT_IDX, INJECT_MARKS):
                if P[i,mi] >= INJECT_THRESH: s += m
            ew.append(s)
        inj.append(ew)
    return inj

def predict_probs_twopass(model, tok, rows):
    """Pass 1 -> inject reliable marks -> Pass 2. Returns (two_pass, single_pass)."""
    p1 = _infer_rows(model, tok, rows, enc_words_list=None)
    inj = _build_injection(rows, p1)
    p2 = _infer_rows(model, tok, rows, enc_words_list=inj)
    out=[]
    for (w1,P1),(w2,P2) in zip(p1,p2):
        if PASS2_BLEND=="p2":
            P=P2
        elif PASS2_BLEND=="mean":
            P=0.5*(P1+P2)
        else:   # "reliable_from_p1": trust pass-1 for injected marks, pass-2 for the rest
            P=P2.copy()
            for mi in INJECT_IDX: P[:,mi]=np.maximum(P1[:,mi], P2[:,mi])
        out.append((w1,P))
    return out, p1

def ensemble_probs(list_of_results):
    base=list_of_results[0]; out=[]
    for k in range(len(base)):
        words=base[k][0]
        P=np.mean([lr[k][1] for lr in list_of_results],axis=0)
        out.append((words,P))
    return out
print("train + single-pass/two-pass inference ready")


## 6. Per-class threshold tuning (free multi-label decode)

In [ ]:
# =========================================================================
#  Per-class threshold search to maximize MACRO-F1.
#  In multi-label the classes are independent, so tuning each mark's threshold
#  separately is optimal for macro-F1. Rare marks get lower thresholds (recall).
# =========================================================================
def per_class_f1(y_true, y_pred):
    tp=((y_pred==1)&(y_true==1)).sum(0)
    fp=((y_pred==1)&(y_true==0)).sum(0)
    fn=((y_pred==0)&(y_true==1)).sum(0)
    prec=tp/np.clip(tp+fp,1,None); rec=tp/np.clip(tp+fn,1,None)
    f1=np.where((prec+rec)>0, 2*prec*rec/np.clip(prec+rec,1e-9,None), 0.0)
    return f1

def tune_thresholds(val_results, val_rows):
    # stack word-level gold + probs across all val rows
    golds=[]; probs=[]
    for (words,P),r in zip(val_results,val_rows):
        _,Y=gold_to_labels(r["final_text"])
        golds.append(Y); probs.append(P)
    Yt=np.concatenate(golds,0); Pp=np.concatenate(probs,0)
    grid=np.linspace(0.10,0.90,33)
    best=np.full(NUM_MARKS,0.5)
    for m in range(NUM_MARKS):
        bf,bt=-1,0.5
        for t in grid:
            pred=(Pp[:,m]>=t).astype(int)
            tp=((pred==1)&(Yt[:,m]==1)).sum()
            fp=((pred==1)&(Yt[:,m]==0)).sum()
            fn=((pred==0)&(Yt[:,m]==1)).sum()
            prec=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
            f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
            if f1>bf: bf,bt=f1,t
        best[m]=bt
    return best

def probs_to_multihot(words, P, thresholds):
    pred=(P>=thresholds[None,:]).astype(np.float32)
    pred=apply_honorific_rule(words,pred)   # force -X- honorifics
    return pred
print("threshold tuning ready")

## 7. Train ensemble & validate (host metric + per-class F1)

In [ ]:
# =========================================================================
#  Orchestrate: train single AraBERT -> compare SINGLE-pass vs TWO-pass on the
#  host metric, with per-class F1 side by side. Thresholds are tuned separately
#  for each decode (they're independent multi-label thresholds).
# =========================================================================
trained=[]
for mn in MODELS:
    print("training", mn)
    model,tok=train_one(mn, train_rows, val_rows)
    trained.append((model,tok))
model, tok = trained[0]     # single-model focus

def _host_macro(results, rows, thresholds):
    texts=[reconstruct(words, probs_to_multihot(words,P,thresholds)) for (words,P) in results]
    vdf=pd.DataFrame([{"id":i,"text":r["text"],"final_text":r["final_text"]} for i,r in enumerate(rows)])
    sdf=pd.DataFrame({"id":range(len(rows)),"final_text":texts})
    return score(vdf.copy(), sdf.copy(), "id")

def _per_class(results, rows, thresholds):
    golds=[]; preds=[]
    for (words,P),r in zip(results,rows):
        _,Y=gold_to_labels(r["final_text"])
        golds.append(Y); preds.append(probs_to_multihot(words,P,thresholds))
    return per_class_f1(np.concatenate(golds), np.concatenate(preds))

# ---------- PASS 1 : single forward pass ----------
val_p1 = predict_probs(model, tok, val_rows)
th1    = tune_thresholds(val_p1, val_rows)
macro_1= _host_macro(val_p1, val_rows, th1)
f1_1   = _per_class(val_p1, val_rows, th1)

# ---------- TWO-PASS : refine with injected reliable marks ----------
if TWO_PASS:
    val_p2, _ = predict_probs_twopass(model, tok, val_rows)
    th2    = tune_thresholds(val_p2, val_rows)
    macro_2= _host_macro(val_p2, val_rows, th2)
    f1_2   = _per_class(val_p2, val_rows, th2)
else:
    th2, macro_2, f1_2 = th1, macro_1, f1_1

print("\n===============  VALIDATION (host macro-F1)  ===============")
print(f"  single pass : {macro_1:.4f}")
print(f"  two  pass   : {macro_2:.4f}   (delta {macro_2-macro_1:+.4f})")
print("\n  per-class F1      single   two-pass    delta")
for i,m in enumerate(MARKS):
    star = "  <- bottleneck" if m in ('؛','!','؟',':') else ""
    print(f"    {m:2s}   {f1_1[i]:8.3f} {f1_2[i]:9.3f}  {f1_2[i]-f1_1[i]:+8.3f}{star}")
print("\n  tuned thresholds (two-pass):", {MARKS[i]:round(float(th2[i]),3) for i in range(NUM_MARKS)})


## 8. Predict test & write submission

In [ ]:
# =========================================================================
#  Predict TEST and write submission.csv  (columns: id, final_text)
#  Uses the TWO-PASS decode when TWO_PASS=True, else single pass.
# =========================================================================
test = pd.read_csv(TEST_CSV)
id_col = "id" if "id" in test.columns else test.columns[0]
test_rows=[{"text":t} for t in test["text"].tolist()]

if TWO_PASS:
    test_res, _ = predict_probs_twopass(model, tok, test_rows)
    thr = th2
else:
    test_res = predict_probs(model, tok, test_rows)
    thr = th1

pred_texts=[reconstruct(words, probs_to_multihot(words,P,thr)) for (words,P) in test_res]
submission=pd.DataFrame({id_col: test[id_col], "final_text": pred_texts})
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv", submission.shape, "| two_pass:", TWO_PASS)
print(submission.head(2).to_string())
